# 03 – Event Replay: 突发事件复盘

This notebook demonstrates the **event study** workflow:
1. Pick a historical market shock (T0)
2. Download price data for multiple assets
3. Compute cumulative returns relative to T0
4. Visualize asset responses on a single chart

**Pre-configured events:**
| Key | Date | Description |
|-----|------|-------------|
| `covid_crash` | 2020-03-09 | COVID-19 circuit-breaker (新冠熔断) |
| `russia_ukraine` | 2022-02-24 | Russia invades Ukraine (俄乌冲突) |
| `fed_hike_2022` | 2022-03-16 | US Fed starts rate-hike cycle (美联储加息) |

In [ ]:
import sys
from pathlib import Path

project_root = Path().resolve().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

In [ ]:
# List available pre-set events
from src.analysis.event_study import list_preset_events

events_df = list_preset_events()
display(events_df)

## Step 1 – Choose an Event

In [ ]:
# Change this to replay a different event
EVENT_KEY = "covid_crash"

PRE_WINDOW  = 30   # days before T0 to include
POST_WINDOW = 90   # days after T0 to include

## Step 2 – Fetch Asset Data

In [ ]:
from src.analysis.event_study import PRESET_EVENTS
from src.data_fetcher.equities import get_index_data
from src.data_fetcher.commodities import get_gold_price
from src.data_fetcher.sentiment import get_vix
import pandas as pd

event = PRESET_EVENTS[EVENT_KEY]
event_date = event.date

# Compute the fetch range with generous buffer
from datetime import date, timedelta
start = str((pd.Timestamp(event_date) - pd.Timedelta(days=PRE_WINDOW + 10)).date())
end   = str((pd.Timestamp(event_date) + pd.Timedelta(days=POST_WINDOW + 10)).date())

print(f"Event : {event.description}")
print(f"T0    : {event_date}")
print(f"Range : {start} → {end}")
print()

price_data = {}

for label, ticker in [("S&P 500", "^GSPC"), ("CSI 300", "000300.SH"), ("Nikkei", "^N225")]:
    print(f"Fetching {label} …")
    df = get_index_data(ticker, start_date=start, end_date=end)
    if not df.empty:
        price_data[label] = df
        print(f"  ✓ {len(df)} rows")
    else:
        print(f"  ✗ No data")

print("Fetching Gold …")
gold = get_gold_price(start_date=start, end_date=end)
if not gold.empty:
    price_data["Gold"] = gold
    print(f"  ✓ {len(gold)} rows")

print("Fetching VIX …")
vix = get_vix(start_date=start, end_date=end)
if not vix.empty:
    price_data["VIX"] = vix
    print(f"  ✓ {len(vix)} rows")

## Step 3 – Run Event Study

In [ ]:
from src.analysis.event_study import analyze_event

if price_data:
    returns = analyze_event(
        event_date=event_date,
        price_data=price_data,
        pre_window=PRE_WINDOW,
        post_window=POST_WINDOW,
    )
    print(f"Event study result shape: {returns.shape}")
    display(returns.head(10))
else:
    print("No price data available.")
    returns = None

## Step 4 – Visualize

In [ ]:
from src.visualization.dashboard_charts import plot_event_impact

if returns is not None and not returns.empty:
    fig = plot_event_impact(
        returns,
        event_date=f"{event_date} ({event.description})",
        backend="plotly",
        title=f"Event Study: {event.description}",
    )
    fig.show()
else:
    print("No returns data to plot.")

## Step 5 – Summary Statistics

In [ ]:
if returns is not None and not returns.empty:
    # Show returns at T0+5, T0+30, T0+60, T0+90 days
    snapshots = [5, 30, 60, 90]
    rows = []
    for days in snapshots:
        idx = returns.index.get_indexer([days], method="nearest")[0]
        row = (returns.iloc[idx] * 100).round(2)
        row.name = f"T0 + {days}d"
        rows.append(row)
    summary = pd.DataFrame(rows)
    print(f"Cumulative returns (%) at key horizons after {event_date}:")
    display(summary)

---
**Try it:** Change `EVENT_KEY` in Step 1 to `"russia_ukraine"` or `"fed_hike_2022"` and re-run all cells to replay a different event.